In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

In [2]:
# 3.- SERIES DE TIEMPO

# 3-3 Zonas Horarias

# Horario central del México tiene 6 horas de diferencia respecto al meridiano de greenwich (Londres)(UTC) 

# Nota
# Cual es la diferencia entre el tipo de datos datetime y timestamp en mysql? 
# En MySQL, tanto el tipo de dato DATETIME como TIMESTAMP se utilizan para almacenar fechas y horas, pero hay algunas diferencias importantes entre ellos:
# DATETIME:
# - Almacena fechas y horas en un formato específico (AAAA-MM-DD HH:MM:SS)
# - No tiene zona horaria asociada
# - No se actualiza automáticamente cuando se inserta o actualiza un registro
# - Puede almacenar fechas y horas en cualquier rango (desde 1000-01-01 00:00:00 hasta 9999-12-31 23:59:59)
# - Requiere 8 bytes de almacenamiento
# TIMESTAMP:
# - Almacena fechas y horas en un formato específico (AAAA-MM-DD HH:MM:SS)
# - Tiene zona horaria asociada (UTC por defecto, es decir, el meridiano de greenwich)
# - Se actualiza automáticamente cuando se inserta o actualiza un registro (si se configura para hacerlo)
# - Puede almacenar fechas y horas en un rango más limitado (desde 1970-01-01 00:00:00 hasta 2038-01-19 03:14:07)
# - Requiere 4 bytes de almacenamiento
# En resumen, la principal diferencia entre DATETIME y TIMESTAMP es que TIMESTAMP tiene zona horaria asociada y se puede configurar para actualizarse 
# automáticamente, mientras que DATETIME no tiene zona horaria asociada y no se actualiza automáticamente.
# Según la experiencia y las prácticas comunes en la industria, el tipo de dato más utilizado en MySQL es DATETIME.
# Hay varias razones por las que DATETIME es más popular:
# 1. Flexibilidad: DATETIME permite almacenar fechas y horas en cualquier rango, desde 1000-01-01 00:00:00 hasta 9999-12-31 23:59:59.
# 2. Simplicidad: DATETIME es más fácil de entender y utilizar que TIMESTAMP, especialmente para aquellos que no están familiarizados con la zona horaria UTC.
# 3. Compatibilidad: DATETIME es compatible con la mayoría de las aplicaciones y frameworks, mientras que TIMESTAMP puede requerir configuraciones adicionales.
# Sin embargo, TIMESTAMP es muy útil en ciertas situaciones, como:
# 1. Almacenamiento de fechas y horas con zona horaria: TIMESTAMP es ideal para almacenar fechas y horas con zona horaria, especialmente cuando se trabaja con aplicaciones globales.
# 2. Actualización automática: TIMESTAMP se puede configurar para actualizarse automáticamente cuando se inserta o actualiza un registro.
# En resumen, aunque DATETIME es el tipo de dato más utilizado en MySQL, TIMESTAMP es muy útil en ciertas situaciones y puede ser la mejor opción dependiendo de las necesidades específicas de tu proyecto.


In [3]:
# Conversion de zona horaria */
# Cuando en londres era 2004-01-01 12:00:00, en Guadalajara era 2004-01-01 06:00:00, ya que son 6 horas hacia atras */
# La funcion Convert_TZ requiere los argumentos: date time, zona horaria en que se registro el dato, zona horaria deseada */

query="""
SELECT 	CONVERT_TZ("2004-01-01 12:00:00","+00:00","-06:00") as HorarioCST; 
"""
df = pd.read_sql(query, engine)
df.head(10)

,HorarioCST
0,2004-01-01 06:00:00


In [4]:
# Conversion de zona horaria */
# Cuando en londres era 2004-01-01 12:00:00, en San Francisco California era 2004-01-01 06:00:00, ya que son 8 horas hacia atras */

query="""
SELECT 	CONVERT_TZ("2004-01-01 12:00:00","+00:00","-08:00") as HorarioCST;
"""
df = pd.read_sql(query, engine)
df.head(10)

,HorarioCST
0,2004-01-01 04:00:00


In [7]:
# /* Conversion de zona horaria de una columna */

query="""
SELECT
    ventasgerentes_id,
    fecha,
    convert_tz(fecha,"+00:00","-6:00") as Horario_Guadalajara
from
    preparacion.ventasgerentes;
"""
df = pd.read_sql(query, engine)
df.head(10)

,ventasgerentes_id,fecha,Horario_Guadalajara
0,1,2012-09-01,2012-08-31 18:00:00
1,2,2012-09-03,2012-09-02 18:00:00
2,3,2012-09-03,2012-09-02 18:00:00
3,4,2012-09-05,2012-09-04 18:00:00
4,5,2012-09-06,2012-09-05 18:00:00
5,6,2012-09-06,2012-09-05 18:00:00
6,7,2012-09-07,2012-09-06 18:00:00
7,8,2012-09-10,2012-09-09 18:00:00
8,9,2012-09-10,2012-09-09 18:00:00
9,10,2012-09-11,2012-09-10 18:00:00


In [5]:
# 3-4 Fechas y Horas

# TimeStamp: Estampa de tiempo */
# MYSQL maneja las fechas en formato YYYY-MM-DD
# Mexico usa las fechas en formato dd-mm-aaaa
# EU usa las fechas en formato mm-dd-aaaa

# Veremos 3 casos: Cambiar el formato de las fechas, extraer partes de la fecha y armar una fecha a partir de sus partes. */


In [6]:
# Obtener fecha actual

query="""
select current_date();
"""
df = pd.read_sql(query, engine)
df.head(10)

,current_date()
0,2026-09-19


In [7]:
# Obtener fecha y hora actual

query="""
select now();
"""
df = pd.read_sql(query, engine)
df.head(10)

,now()
0,2026-09-19 22:52:46


In [32]:
# date_format() es una función de MySQL que permite mostrar una fecha o un valor de fecha y hora en un formato específico.
# Recibe dos argumentos: la fecha a formatear y el formato deseado.
# El primer argumento es la fecha a formatear, que puede ser una fecha literal, una columna de tipo fecha o un valor de fecha y hora. 
# Generalmente MySQL reconoce automaticamente el primer argumento como una fecha, pero si no lo hace, podemos tranformar el primer argumento a un tipo de dato fecha/hora usando la función STR_TO_DATE() que veremos más adelante.
# Retorna un valor de tipo cadena de texto (string) con la fecha formateada según el segundo argumento.

# Código	y su significado
# %Y	Año con 4 dígitos
# %m	Mes con 2 dígitos
# %d	Día con 2 dígitos
# %H	Hora en formato 24 horas
# %h	Hora en formato 12 horas
# %i	Minutos
# %s	Segundos

# En resumen
# STR_TO_DATE(): Recibe texto, nosotros le indicamos como interpretar la cadena y retorna un tipo de dato fecha/hora
# DATE_FORMAT(): Recibe un tipo de dato fecha/hora y retorna texto en el formato de fecha/hora que le indiquemos

# Mostrar una fecha en formato Año-Mes-Día
query="""
select date_format("2020-12-17","%Y-%m-%d") as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [17]:
# Convertir una cadena de texto a tipo de dato fecha/hora

query="""
select str_to_date('17/12/2020 14:15:16','%d/%m/%Y %H:%i:%s') as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head()

,fecha
0,2020-12-17 14:15:16


In [25]:
# Mostrar una fecha en formato Año-Mes-Día

query="""
select date_format("2020-12-17","%y-%M-%D");
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"date_format(""2020-12-17"",""%y-%M-%D"")"
0,20-December-17th


In [26]:
# Mostrar una fecha en formato Año-Mes-Día con Día=01

query="""
select date_format("2020-12-17","%Y-%m-01");
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"date_format(""2020-12-17"",""%Y-%m-01"")"
0,2020-12-01


In [27]:
# Convertir un datetime a date

query="""
select date("2020-12-17 14:00:00") as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [28]:
#  Extraer el dia, mes, año, hora, minuto, segundo de un datetime

query="""
select
    "2020-12-17 14:15:20" as fecha,
    extract(day from "2020-12-17 14:15:20") 	as dia,
    extract(month from "2020-12-17 14:15:20") 	as mes,
    extract(year from "2020-12-17 14:15:20") 	as año,
    extract(hour from "2020-12-17 14:15:20") 	as hora,
    extract(minute from "2020-12-17 14:15:20") 	as minutos,
    extract(second from "2020-12-17 14:15:20") 	as segundos;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha,dia,mes,año,hora,minutos,segundos
0,2020-12-17 14:15:20,17,12,2020,14,15,20


In [30]:
# Imprimir una fecha y hora a partir de cadenas de texto

query="""
select date_format(concat("2020-12-17"," ","14:15:16"),"%Y-%m-%d %h:%m:%s") as fecha_hora; 
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha_hora
0,2020-12-17 02:12:16


In [31]:
# Crear una fecha a partir de cadenas de texto 

quer="""
select date_format(concat("2020-12-17"),"%Y-%m-%d") as fecha; 
"""
df = pd.read_sql(text(quer), engine)
df.head(10)

,fecha
0,2020-12-17


In [36]:
# Crear fecha a partir del año y cuantos días han transcurrido

query="""
select makedate(2020,352) as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [37]:
# Crear una hora a partir de hora, minuto y segundos

query="""
select maketime(14,15,20);
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"maketime(14,15,20)"
0,0 days 14:15:20


In [40]:
# 3-5-Date Math

# Calcular la diferencia entre 2 fechas y expresarla en días, meses, años, horas, minutos y segundos

query="""
select
    timestampdiff(DAY, "2020-01-01","2020-02-01") 	as diferencia_en_dias,
    timestampdiff(MONTH, "2020-01-01","2020-02-01") as diferencia_en_meses,
    timestampdiff(YEAR, "2020-01-01","2020-02-01") 	as diferencia_en_años,
    timestampdiff(HOUR, "2020-01-01","2020-02-01") 	as diferencia_en_horas,
    timestampdiff(MINUTE, "2020-01-01","2020-02-01") as diferencia_en_minutos,
    timestampdiff(SECOND, "2020-01-01","2020-02-01") as diferencia_en_segundos;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,diferencia_en_dias,diferencia_en_meses,diferencia_en_años,diferencia_en_horas,diferencia_en_minutos,diferencia_en_segundos
0,31,1,0,744,44640,2678400


In [ ]:
# Obtener una fecha a partir de una fecha dada y un intervalo

query="""
SELECT DATE_ADD("2017-12-01", interval 16 day ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2017-12-17


In [43]:
# Obtener una fecha a partir de una fecha dada y un intervalo

query="""
SELECT DATE_ADD("2017-12-01", interval 1 month ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2018-01-01


In [45]:
# Obtener una fecha a partir de una fecha dada y un intervalo */

query="""
SELECT DATE_ADD("2017-12-01", interval 5 second ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)


,nueva_fecha
0,2017-12-01 00:00:05


In [46]:
# Obtener una fecha a partir de una fecha dada y un intervalo */

query="""
SELECT DATE_ADD("2017-12-01", interval -1 day ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2017-11-30


In [49]:
# Es posible sumar fechas y multiples intervalos sin usar la funcion date_add */

query="""
select date("2020-01-01") + interval 1 day + interval 1 hour - interval 15 minute + interval 15 second as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2020-01-02 00:45:15


In [51]:
# Podemos usar funciones de fecha al filtrar datos

query="""
select
    fecha,
    sexo 
from
    preparacion.ventasgerentes
where fecha > date("2013-01-01") + interval 30 day
order by fecha asc;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha,sexo
0,2013-02-01,M
1,2013-02-02,H
2,2013-02-02,H
3,2013-02-02,H
4,2013-02-03,M
5,2013-02-06,H
6,2013-02-06,H
7,2013-02-06,H
8,2013-02-06,M
9,2013-02-07,H
